# sklearn model with tensorflow keras tuner

In [ ]:
# ! python -m pip install --no-index --find-links=/kaggle/usr/lib/pip_install_permanent/my_packages -r /kaggle/usr/lib/pip_install_permanent/requirements.txt

In [ ]:
import os
os.environ['PACKAGE_DIR'] = '/kaggle/usr/lib/pip_install_permanent'

In [ ]:
from helper_func import *
# import helper_functions as hf
import sys
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())
from spellchecker import SpellChecker

from tqdm.contrib.concurrent import process_map  # If this import fails, you might need to update tqdm

import multiprocessing
 
# Import Packages
# import shutup; shutup.please()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag, ne_chunk
from textblob import TextBlob

from textstat import flesch_reading_ease, smog_index

import spacy
from collections import Counter
from gensim import corpora, models
import pyLDAvis.gensim as gen
import pyLDAvis
import re

# Machine Learning & Data Preprocessing

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Deep Learning

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gensim
# from gensim.models import Word2Vec, KeyedVectors
import pandas as pd
# Progress bar
from tqdm import tqdm

# Keras Tuner
from keras_tuner.tuners import RandomSearch

# # Setting logging levels and environment variables
# tf.get_logger().setLevel(logging.ERROR)
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from textstat import flesch_reading_ease

In [ ]:
# from helper_functions import *
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())
print('Packages Instaled......')

In [ ]:
train = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv')


In [ ]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

In [ ]:
train['misspelling_count'] = train['clean_text'].apply(count_misspellings)

In [ ]:
from textstat import flesch_reading_ease, smog_index 

train = add_text_features(train, 'full_text', status= 'Pre')

In [ ]:
train

In [ ]:
text_col = 'clean_text'   # 'segmented_text'

glove_path = '/kaggle/input/embeddings/glove-840B-300d.txt'
paragran_path = '/kaggle/input/embeddings/paragram-300-sl999.txt'
fastetxt_path = '/kaggle/input/embeddings/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)

In [ ]:
misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

In [ ]:
# Example usage
corrected_words, uncorrected_words = main(misspellings)

# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))

In [ ]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

# Switch add_features to after segmentation????

In [ ]:
# Apply the parallelization

train = parallelize_dataframe(train, apply_segmentation)

train.head()

In [ ]:
# text_col = 'segmented_text'  


# glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
# paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
# fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# # Rebuild and check vocab after cleaning contractions

# train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)


# misspellings = []

# oov = glove + paragram + fastetxt

# for word, _ in oov:

#     misspellings.append(word)

#     misspellings = list(set(misspellings))

# print(f"Number of misspelled words: {len(misspellings)}")

# # print(f"Misspelled words: {misspellings}")


# # Example usage
# corrected_words, uncorrected_words = main(misspellings)

# # Assuming df is your DataFrame and 'clean_text' is the column you want to correct

# correction_dict = dict(corrected_words)

# train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))

# print(f"Corrected words: {len(corrected_words)}")
# print(f"Uncorrected words: {len(uncorrected_words)}")

In [ ]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

train_essays, _ = preprocess_data(train_essays)

validation_essays, _ = preprocess_data(validation_essays) #, tfidf_vectorizer=tfidf_vectorizer)

In [ ]:
train_essays.to_parquet('sklearn_train_essays.parquet')
validation_essays.to_parquet('sklearn_validation_essays.parquet')

In [ ]:
def count_tags(pos_list):
    """ Count the frequency of POS tags from a list of tuples with (word, tag). """
    tag_counts = {}
    for _, tag in pos_list:
        if tag in tag_counts:
            tag_counts[tag] += 1
        else:
            tag_counts[tag] = 1
    return tag_counts

# # Apply the function to each row's pos_tags to create a new column of dictionaries
# train['tag_counts'] = train['pos_tags'].apply(count_tags)

# # Convert the dictionaries into a DataFrame
# result = pd.json_normalize(train['tag_counts'])

# # Merge the tag DataFrame with the original DataFrame
# train = pd.concat([train, result], axis=1).fillna(0)

# train.head()


In [2]:
import pandas as pd

train_essays = pd.read_parquet('/home/jack/github/kaggle/scoring/train_essays.parquet')
validation_essays = pd.read_parquet('/home/jack/github/kaggle/scoring/validation_essays.parquet')

In [3]:



# # Apply the function to each row's pos_tags to create a new column of dictionaries
# train_essays['tag_counts'] = train_essays['Pre_pos_tags'].apply(count_tags)

# # Convert the dictionaries into a DataFrame
# result = pd.json_normalize(train_essays['tag_counts'])

# # Merge the tag DataFrame with the original DataFrame
# train_essays = pd.concat([train_essays, result], axis=1).fillna(0)






# # Apply the function to each row's pos_tags to create a new column of dictionaries
# validation_essays['tag_counts'] = validation_essays['Pre_pos_tags'].apply(count_tags)

# # Convert the dictionaries into a DataFrame
# result = pd.json_normalize(validation_essays['tag_counts'])

# # Merge the tag DataFrame with the original DataFrame
# validation_essays = pd.concat([validation_essays, result], axis=1).fillna(0)



In [4]:
train_essays.head()

,essay_id,full_text,score,lowered,clean_text,misspelling_count,corrected_text,segmented_text,paragraph_count,sentence_count,...,tfidf_13798,tfidf_13799,tfidf_13800,tfidf_13801,tfidf_13802,tfidf_13803,tfidf_13804,tfidf_13805,tfidf_13806,tfidf_13807
1,000fe60,I am a scientist at NASA that is discussing th...,3,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...,9,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...,1,1,...,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0
2,001ab80,People always wish they had the same technolog...,4,people always wish they had the same technolog...,people always wish they had the same technolog...,8,people always wish they had the same technolog...,people always wish they had the same technolog...,1,1,...,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3,"dear, state senator\n\nthis is a letter to arg...",dear state senator this is a letter to argue i...,9,dear state senator this is a letter to argue i...,dear state senator this is a letter to argue i...,1,1,...,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0
6,0033037,The posibilty of a face reconizing computer wo...,2,the posibilty of a face reconizing computer wo...,the posibilty of a face reconizing computer wo...,6,the pos i bil ty of a face recon i zing comput...,the pos i bil ty of a face recon i zing comput...,1,1,...,0.030411,0.0,0.0,0.0,0.061333,0.062338,0.067106,0.067389,0.0,0.0
7,0033bf4,What is the Seagoing Cowboys progam?\n\nIt was...,3,what is the seagoing cowboys progam?\n\nit was...,what is the seagoing cowboys progam it was to ...,4,what is the seagoing cowboys prog am it was to...,what is the seagoing cowboys prog am it was to...,1,1,...,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0


In [5]:
validation_essays.head()

,essay_id,full_text,score,lowered,clean_text,misspelling_count,corrected_text,segmented_text,paragraph_count,sentence_count,...,tfidf_13798,tfidf_13799,tfidf_13800,tfidf_13801,tfidf_13802,tfidf_13803,tfidf_13804,tfidf_13805,tfidf_13806,tfidf_13807
0,000d118,Many people have car where they live. The thin...,3,many people have car where they live. the thin...,many people have car where they live the thing...,24,many people have car where they live the thing...,many people have car where they live the thing...,1,1,...,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.018737,0.01947
3,001bdc0,"We all heard about Venus, the planet without a...",4,"we all heard about venus, the planet without a...",we all heard about venus the planet without al...,7,we all heard about venus the planet without al...,we all heard about venus the planet without al...,1,1,...,0.017602,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00000
5,0030e86,If I were to choose between keeping the electo...,4,if i were to choose between keeping the electo...,if i were to choose between keeping the electo...,12,if i were to choose between keeping the electo...,if i were to choose between keeping the electo...,1,1,...,0.000000,0.05125,0.045166,0.045246,0.0,0.0,0.0,0.0,0.000000,0.00000
8,0036253,The challenge of exploring Venus\n\nThis stori...,2,the challenge of exploring venus\n\nthis stori...,the challenge of exploring venus this storie i...,25,the challenge of exploring venus this s to rie...,the challenge of exploring venus this s to rie...,1,1,...,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00000
14,0065bd6,Driverless cars should not exsist it can cause...,3,driverless cars should not exsist it can cause...,driverless cars should not exsist it can cause...,3,driverless cars should not ex s is t it can ca...,driverless cars should not ex s is t it can ca...,1,1,...,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00000


In [6]:
import pandas as pd
from sklearn.utils import resample

def sample_df(df, col='score', sub=0, random_state=None):
    """
    Balances the classes in a DataFrame by resampling.
    
    Parameters:
    - df: DataFrame to be resampled.
    - col: The column name in df that contains class labels.
    - random_state: The random state for reproducibility.
    
    Returns:
    - balanced_df: A DataFrame with balanced classes.
    """
    class_counts = df[col].value_counts()
    target_count = max(int(class_counts.median()) - sub, class_counts.min())  # Ensure target_count is positive
    
    balanced_df = pd.DataFrame()

    for class_label in df[col].unique():
        class_subset = df[df[col] == class_label]
        
        if len(class_subset) > target_count:
            # Downsample majority classes
            class_subset = resample(class_subset,
                                    replace=False,
                                    n_samples=target_count,
                                    random_state=random_state)
        else:
            # Upsample minority classes
            class_subset = resample(class_subset,
                                    replace=True,
                                    n_samples=target_count,
                                    random_state=random_state)
        
        balanced_df = pd.concat([balanced_df, class_subset], axis=0)
    
    # Shuffle the DataFrame to mix the classes well
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return balanced_df


In [7]:
# Balance only the training DataFrame
sampled_train_essays = sample_df(train_essays, col='score', random_state=42, sub=1000)

sampled_val_essays = sample_df(validation_essays, col='score', sub=0, random_state=42)

In [8]:
print(sampled_train_essays['score'].value_counts())
print(sampled_val_essays['score'].value_counts())

score
5    936
2    936
3    936
6    936
4    936
1    936
Name: count, dtype: int64
score
3    653
6    653
4    653
5    653
1    653
2    653
Name: count, dtype: int64


In [9]:
sampled_val_essays.columns

Index(['essay_id', 'full_text', 'score', 'lowered', 'clean_text',
       'misspelling_count', 'corrected_text', 'segmented_text',
       'paragraph_count', 'sentence_count',
       ...
       'tfidf_13798', 'tfidf_13799', 'tfidf_13800', 'tfidf_13801',
       'tfidf_13802', 'tfidf_13803', 'tfidf_13804', 'tfidf_13805',
       'tfidf_13806', 'tfidf_13807'],
      dtype='object', length=13831)

In [10]:
STATUS = 'Post'

if STATUS == 'Post':
    
    drop_cols = [ 'full_text', 'lowered', 'clean_text','corrected_text', 'segmented_text']

# else:
    
#     drop_cols = ['full_text', 'lowered', 'clean_text',
#         'Pre_tokens', 'Pre_sentences', 'Pre_pos_tags',
#         'corrected_text', 'segmented_text',]



train_df = sampled_train_essays.copy()
val_df = sampled_val_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
val_df.drop(columns=drop_cols, inplace= True)

In [11]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [12]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
import numpy as np

# Assuming 'feature_cols' are defined elsewhere in your script

# Labels (stays the same)
sampled_train_labels = np.array(train_df['score'])
sampled_val_labels = np.array(val_df['score'])
full_train_labels = np.array(train_essays['score'])
full_val_labels = np.array(validation_essays['score'])

# sampled_train_labels = pd.DataFrame(sampled_train_labels, columns=['score'],
#                              index=sampled_train_labels.index)

# sampled_val_labels = pd.DataFrame(sampled_val_labels, columns=['score'], 
#                           index=sampled_val_labels.index)


# full_train_labels = pd.DataFrame(full_train_labels, columns=['score'],
#                              index=full_train_labels.index)

# full_val_labels = pd.DataFrame(full_val_labels, columns=['score'], 
#                           index=full_val_labels.index)



# Features
sampled_train_features = train_df[feature_cols]
sampled_val_features = val_df[feature_cols]
full_train_features = train_essays[feature_cols]
full_val_features = validation_essays[feature_cols]

# Initialize scaler
scaler = StandardScaler()

# Fit scaler to the original full training data
scaler.fit(full_train_features)

# Transform all datasets using the fitted scaler
sampled_train_feats_scaled = scaler.transform(sampled_train_features)
sampled_val_feats_scaled = scaler.transform(sampled_val_features)
full_train_feats_scaled = scaler.transform(full_train_features)
full_val_feats_scaled = scaler.transform(full_val_features)

# Save the scaler for later use
with open('sklearn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [13]:
# Check if the file has been written correctly and is not empty
import os
scaler_path = 'sklearn_scaler.pkl'
if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

Scaler saved successfully in sklearn_scaler.pkl.


In [14]:
# Check if the scaler is StandardScaler
if isinstance(scaler, StandardScaler):
    print("The scaler is a StandardScaler.")
elif isinstance(scaler, MinMaxScaler):
    print("The scaler is a MinMaxScaler.")
else:
    print("The scaler is neither StandardScaler nor MinMaxScaler.")

The scaler is a StandardScaler.


In [15]:
# import keras_tuner
# from sklearn import ensemble
# from sklearn import linear_model
# from sklearn import model_selection
# from sklearn import metrics
# from sklearn.metrics import make_scorer, cohen_kappa_score

# def build_model(hp):
#     """
#     Builds a machine learning model based on hyperparameters.
    
#     Parameters:
#     hp : HyperParameters
#         Hyperparameters for tuning the model.
    
#     Returns:
#     model : An instance of a Scikit-learn model.
#     """
#     model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
#     if model_type == 'random_forest':
#         model = ensemble.RandomForestClassifier(
#             n_estimators=hp.Int('n_estimators', 10, 50, step=10),
#             max_depth=hp.Int('max_depth', 3, 10),
#             min_samples_split=hp.Int('min_samples_split', 2, 10),
#             min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
#             criterion=hp.Choice('criterion', ['gini', 'entropy']),
#             class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
#             max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log'))
#     else:
#         model = linear_model.RidgeClassifier(
#             alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))

#     return model

def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')


# # Wrapping QWK as a custom scorer for model evaluation
# qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

# # Tuner configuration
# tuner = keras_tuner.tuners.SklearnTuner(
#     oracle=keras_tuner.oracles.BayesianOptimizationOracle(
#         objective=keras_tuner.Objective('score', 'max'),
#         max_trials=100),
#     hypermodel=build_model,
#     scoring=qwk_scorer,  # Use the QWK scorer
#     cv=model_selection.StratifiedKFold(10),
#     directory='.',
#     project_name='data/sklearn',
#     overwrite=True)


# tuner.search(train_features, train_labels)

# best_model = tuner.get_best_models(num_models=1)[0]

In [16]:
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
# import keras_tuner as kt

# model_checkpoint = ModelCheckpoint('data/models/best_standard_model_epoch.keras', 
#                                    save_best_only=True, monitor='val_loss', mode='min')

# early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
#                                restore_best_weights=True)

# call_backs = [model_checkpoint, early_stopping]

In [18]:
import keras_tuner
from sklearn import ensemble, linear_model, model_selection, svm
from sklearn.metrics import make_scorer, cohen_kappa_score

def build_model(hp):
    """
    Builds a more comprehensive machine learning model based on hyperparameters for multiclass classification.
    
    Parameters:
    hp : HyperParameters
        Hyperparameters for tuning the model.
    
    Returns:
    model : An instance of a Scikit-learn model.
    """
    model_type = hp.Choice('model_type', ['random_forest', 'ridge', 'svm', 'gradient_boosting'])
    
    if model_type == 'random_forest':
        model = ensemble.RandomForestClassifier(
            n_estimators=hp.Int('n_estimators', 10, 100, step=10),
            max_depth=hp.Int('max_depth', 3, 20),
            min_samples_split=hp.Int('min_samples_split', 2, 20),
            min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
            criterion=hp.Choice('criterion', ['gini', 'entropy']),
            class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
            max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log'))
    elif model_type == 'ridge':
        model = linear_model.RidgeClassifier(
            alpha=hp.Float('alpha', 1e-4, 10, sampling='log'))
    elif model_type == 'svm':
        model = svm.SVC(
            C=hp.Float('C', 1e-4, 10, sampling='log'),
            kernel=hp.Choice('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
            class_weight='balanced',
            probability=True)  # For supporting probability estimation
    elif model_type == 'gradient_boosting':
        model = ensemble.GradientBoostingClassifier(
            n_estimators=hp.Int('n_estimators', 50, 200, step=50),
            learning_rate=hp.Float('learning_rate', 0.01, 0.2, sampling='log'),
            max_depth=hp.Int('max_depth', 3, 15),
            subsample=hp.Float('subsample', 0.5, 1.0, step=0.1))

    return model

# Defining the custom QWK scorer
qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

# Tuner configuration
tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=20),
    hypermodel=build_model,
    scoring=qwk_scorer,
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='data/sklearn',
    overwrite=True)

# Starting the search
tuner.search(full_train_feats_scaled, full_train_labels)      # class_weight=class_weights_dict,

# Retrieving the best model


Trial 7 Complete [14h 05m 19s]
score: 0.5213163043600778

Best score So Far: 0.6339343272048598
Total elapsed time: 14h 38m 03s

Search: Running Trial #8

Value             |Best Value So Far |Hyperparameter
svm               |random_forest     |model_type
20                |70                |n_estimators
17                |10                |max_depth
2                 |18                |min_samples_split
6                 |8                 |min_samples_leaf
gini              |entropy           |criterion
balanced_subsample|balanced          |class_weight
0.20269           |0.9554            |max_samples
0.88022           |8.1839            |alpha
8.8705            |None              |C
poly              |None              |kernel



In [ ]:
# train_labels['score'].unique()
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.fit(full_train_feats_scaled, full_train_labels)

In [ ]:
predictions = best_model.predict(full_val_features)

y_true = full_val_labels

# predictions = target_scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()


In [ ]:
# confusion matrix

from sklearn.metrics import confusion_matrix

confusion_matrix(y_true, predictions)

In [ ]:
# classification report

from sklearn.metrics import classification_report

print(classification_report(y_true, predictions))

In [ ]:
# cohens kappa

from sklearn.metrics import cohen_kappa_score

cohen = cohen_kappa_score(y_true, predictions)

In [ ]:
# quadratic weighted kappa

from sklearn.metrics import cohen_kappa_score

quadratic = cohen_kappa_score(y_true, predictions, weights='quadratic')

print(f"Cohen's Kappa: {cohen}")
print(f"Quadratic Weighted Kappa: {quadratic}")

In [ ]:
from joblib import dump, load

dump(best_model, 'standard_random_forest.joblib') 

In [ ]:
# forest_model = load('random_forest.joblib') 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generate and display the confusion matrix for the test predictions
cm = confusion_matrix(y_true, predictions, labels=best_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()
